# Spark ML Vector Search – Table Profiling

Profile `BucketedRandomProjectionLSH` ANN search against a selectable source table.

For the selected table the notebook:
1. Converts every row into a Spark ML vector
2. Fits LSH on the full table
3. Runs a self-join ANN query so every row gets its 10 closest neighbors
4. Reports fit time, search time, and distance statistics

*Tested on ML Runtime 15.4 LTS*

In [0]:
import time
import pandas as pd
from pyspark.sql import functions as F, Window
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.functions import array_to_vector

# --- Configuration (widget-driven) ---
SOURCE_TABLE = dbutils.widgets.get("source_table")
TOP_K = 10
BUCKET_LENGTH = 2.0          # larger bucket = fewer candidates per bucket (faster, slightly less recall)
NUM_HASH_TABLES = 3          # fewer tables = fewer candidate pairs to evaluate
DISTANCE_THRESHOLD = 2.0     # tight threshold prunes most pairs early (p90 dist ~0.76 at 1k)
NUM_PARTITIONS = 64          # parallelism for the self-join

# --- Load and cache base table ---
full_df = (
    spark.table(SOURCE_TABLE)
    .select(
        "registeredAccountId",
        array_to_vector("query_vec").alias("features")
    )
    .where(F.col("features").isNotNull())
    .repartition(NUM_PARTITIONS)
    .cache()
)

total_rows = full_df.count()  # materializes the cache
print(f"Source table: {SOURCE_TABLE}")
print(f"Total rows: {total_rows:,} (cached in {NUM_PARTITIONS} partitions)")
print(f"LSH params: bucketLength={BUCKET_LENGTH}, numHashTables={NUM_HASH_TABLES}")
print(f"TOP_K={TOP_K}, distanceThreshold={DISTANCE_THRESHOLD}")

## Table profiling

Fit LSH on the selected table, then run a self-join so every row in the table gets its 10 closest neighbors. This performs exactly one ANN inference per row in the selected table.

In [0]:
print(f"\n{'='*60}")
print(f"Profiling table: {SOURCE_TABLE}")
print(f"{'='*60}")

# Fit LSH on the full selected table
fit_start = time.perf_counter()
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=BUCKET_LENGTH,
    numHashTables=NUM_HASH_TABLES,
)
lsh_model = lsh.fit(full_df)
fit_ms = (time.perf_counter() - fit_start) * 1000
print(f"Fit: {fit_ms:.1f}ms")

# Precompute hashes once so the self-join can reuse them on both sides
hashed_df = (
    lsh_model.transform(full_df)
    .select("registeredAccountId", "features", "hashes")
    .cache()
)
_ = hashed_df.count()

# Self-join: every row finds its TOP_K closest neighbors
search_start = time.perf_counter()
raw_matches = (
    lsh_model.approxSimilarityJoin(
        hashed_df,
        hashed_df,
        DISTANCE_THRESHOLD,
        distCol="distCol",
    )
    .select(
        F.col("datasetA.registeredAccountId").alias("query_id"),
        F.col("datasetB.registeredAccountId").alias("neighbor_id"),
        F.col("distCol").alias("distance"),
    )
    .where(F.col("query_id") != F.col("neighbor_id"))
)

# Keep TOP_K neighbors per query without a window sort
ranked = (
    raw_matches
    .groupBy("query_id")
    .agg(
        F.slice(
            F.array_sort(
                F.collect_list(
                    F.struct(
                        F.col("distance").alias("distance"),
                        F.col("neighbor_id").alias("neighbor_id"),
                    )
                )
            ),
            1,
            TOP_K,
        ).alias("neighbors")
    )
    .cache()
)

queries_with_results = ranked.count()
search_ms = (time.perf_counter() - search_start) * 1000
result_count = ranked.select(F.coalesce(F.sum(F.size("neighbors")), F.lit(0)).alias("result_rows")).first()["result_rows"]

distance_df = (
    ranked
    .select(F.explode("neighbors").alias("neighbor"))
    .select(F.col("neighbor.distance").alias("distance"))
)

if result_count > 0:
    p50, p90, p99 = distance_df.approxQuantile("distance", [0.50, 0.90, 0.99], 0.001)
else:
    p50, p90, p99 = (None, None, None)

profile_row = {
    "source_table": SOURCE_TABLE,
    "rows_profiled": total_rows,
    "fit_ms": round(fit_ms, 1),
    "search_ms": round(search_ms, 1),
    "total_ms": round(fit_ms + search_ms, 1),
    "result_rows": result_count,
    "queries_with_results": queries_with_results,
    "avg_neighbors": round(result_count / max(queries_with_results, 1), 2),
    "p50_distance": round(p50, 4) if p50 is not None else None,
    "p90_distance": round(p90, 4) if p90 is not None else None,
    "p99_distance": round(p99, 4) if p99 is not None else None,
}

print(f"Search: {search_ms:.1f}ms | Results: {result_count:,} | Queries w/ results: {queries_with_results:,}")
print(f"Avg neighbors/query: {profile_row['avg_neighbors']} | p50 dist: {profile_row['p50_distance']} | p90 dist: {profile_row['p90_distance']}")
print(f"\n{'='*60}")
print("Profiling complete.")

## Results summary

In [0]:
profile_df = pd.DataFrame([profile_row])
print(profile_df.to_string(index=False))
display(spark.createDataFrame(profile_df))